# Bronze Layer — Ingestion 
**CNG Distribution Analytics Platform**

Reads raw CSV files, stamps audit columns, runs data quality checks,
and writes as CSV files to `Files/bronzedata/`.


| Audit Column | Description |
|---|---|
| `_ingested_at` | Timestamp when this notebook ran |
| `_source_file` | Original CSV filename |
| `_pipeline_run_id` | Pipeline run ID |

## 1. Config

In [14]:
import os
import pandas as pd

# ── Paths ────────────────────────────────────────────────────────────────────
# Point SOURCE_PATH at wherever your CSVs live
SOURCE_PATH = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/RawData/"          # input  — your raw CSVs
OUTPUT_PATH = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/"   # output — parquet files

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── Pipeline run ID ──────────────────────────────────────────────────────────
# In Fabric this is passed in from Data Factory at runtime
# Locally we generate a timestamp-based ID
from datetime import datetime
ingested_at    = datetime.now()
pipeline_run_id = ingested_at.strftime('%Y%m%d_%H%M%S')

# ── File config ───────────────────────────────────────────────────────────────
FILE_CONFIGS = [
    {
        "source_file"  : "PurchaseOrders.csv",
        "target"       : "bronze_purchase_orders",
        "required_cols": ["POID", "ProductID", "SupplierID"],
        "pk"           : "POID"
    },
    {
        "source_file"  : "SalesOrders.csv",
        "target"       : "bronze_sales_orders",
        "required_cols": ["OrderID", "CustomerID", "ProductID"],
        "pk"           : "OrderID"
    },
    {
        "source_file"  : "Products.csv",
        "target"       : "bronze_products",
        "required_cols": ["ProductID"],
        "pk"           : "ProductID"
    },
    {
        "source_file"  : "Customers.csv",
        "target"       : "bronze_customers",
        "required_cols": ["CustomerID"],
        "pk"           : "CustomerID"
    },
    {
        "source_file"  : "Inventory.csv",
        "target"       : "bronze_inventory",
        "required_cols": ["ProductID", "WarehouseID"],
        "pk"           : "ProductID"
    }
]

print(f"Config loaded  : {len(FILE_CONFIGS)} files")
print(f"Source path    : {SOURCE_PATH}")
print(f"Output path    : {OUTPUT_PATH}")
print(f"Run ID         : {pipeline_run_id}")
print(f"Ingested at    : {ingested_at.strftime('%Y-%m-%d %H:%M:%S')}")

Config loaded  : 5 files
Source path    : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/RawData/
Output path    : abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/
Run ID         : 20260604_211529
Ingested at    : 2026-06-04 21:15:29


## 3. Write function


In [15]:
def write_bronze(df: pd.DataFrame, target: str):
    """
    writes csv to Files/bronzedata/
    """
    out_path = OUTPUT_PATH + target + ".csv"
    df.to_csv(out_path, index=False)
    print(f"  Written to: {out_path}")

## 4. Data quality check function

In [16]:
def run_quality_checks(df: pd.DataFrame, cfg: dict) -> list:
    """
    Runs data quality checks on the ingested DataFrame.
    Returns list of issues. Empty list = all passed.
    Bronze logs issues but does NOT fail — Silver acts on them.
    """
    issues = []
    total_rows = len(df)
    print(f"  ── Quality checks ──")

    # Check 1: Row count
    if total_rows == 0:
        msg = "FAIL: Table is empty"
        print(f"  {msg}")
        issues.append(msg)
    else:
        print(f"  [OK ] Row count: {total_rows:,}")

    # Check 2: Null % on required columns(primary key)
    for col_name in cfg["required_cols"]:
        if col_name in df.columns:
            null_count = df[col_name].isna().sum()
            null_pct   = null_count / total_rows if total_rows > 0 else 0
            status     = "FAIL" if null_pct > 0.1 else "WARN" if null_pct > 0 else "OK "
            print(f"  [{status}] {col_name} nulls: {null_pct:.1%} ({null_count:,} rows)")
            if status == "FAIL":
                issues.append(f"{col_name} has {null_pct:.1%} nulls")
        else:
            msg = f"FAIL: Required column '{col_name}' not found in file"
            print(f"  {msg}")
            issues.append(msg)

    # Check 3: Duplicate primary key
    pk = cfg["pk"]
    if pk in df.columns:
        dupe_count = total_rows - df[pk].nunique()
        status     = "WARN" if dupe_count > 0 else "OK "
        print(f"  [{status}] Duplicate {pk}: {dupe_count:,}")
        if dupe_count > 0:
            issues.append(f"{dupe_count:,} duplicate {pk} values")

    # Check 4: Audit columns present
    audit_cols = ["_ingested_at", "_source_file", "_pipeline_run_id"]
    missing_audit = [c for c in audit_cols if c not in df.columns]
    if missing_audit:
        msg = f"FAIL: Missing audit columns: {missing_audit}"
        print(f"  {msg}")
        issues.append(msg)
    else:
        print(f"  [OK ] All audit columns present")

    return issues

## 5. Ingest all files

In [17]:
results = []

for cfg in FILE_CONFIGS:
    print(f"\n{'='*55}")
    print(f"Processing: {cfg['source_file']}")
    print(f"{'='*55}")

    try:
        file_path = SOURCE_PATH + cfg["source_file"]

        # Pre-check: file exists
        #if not os.path.exists(file_path):
        #    raise FileNotFoundError(f"File not found: {file_path}")

    
        df = pd.read_csv(file_path, dtype=str)
        print(f"  Read {len(df):,} rows, {len(df.columns)} cols")

        # Stamp audit columns — the ONLY thing added in Bronze
        df["_ingested_at"]     = str(ingested_at)
        df["_source_file"]     = cfg["source_file"]
        df["_pipeline_run_id"] = pipeline_run_id

        # Write to output
        write_bronze(df, cfg["target"])

        # Data quality checks
        issues = run_quality_checks(df, cfg)

        results.append({
            "file"  : cfg["source_file"],
            "target": cfg["target"] + ".csv",
            "status": "SUCCESS",
            "rows"  : len(df),
            "issues": "; ".join(issues) if issues else "None"
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({
            "file"  : cfg["source_file"],
            "target": cfg["target"] + ".csv",
            "status": "FAILED",
            "rows"  : 0,
            "issues": str(e)
        })

print(f"\n{'='*55}")
print("BRONZE INGESTION COMPLETE")
print(f"{'='*55}")



Processing: PurchaseOrders.csv
  Read 1,248 rows, 19 cols
  Written to: abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/bronze_purchase_orders.csv
  ── Quality checks ──
  [OK ] Row count: 1,248
  FAIL: Required column 'PONumber' not found in file
  [WARN] ProductID nulls: 1.9% (24 rows)
  [WARN] SupplierID nulls: 2.8% (35 rows)
  [OK ] All audit columns present

Processing: SalesOrders.csv
  Read 5,150 rows, 17 cols
  Written to: abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/bronze_sales_orders.csv
  ── Quality checks ──
  [OK ] Row count: 5,150
  [OK ] OrderID nulls: 0.0% (0 rows)
  [OK ] CustomerID nulls: 0.0% (0 rows)
  [WARN] ProductID nulls: 2.8% (144 rows)
  [WARN] Duplicate OrderID: 87
  [OK ] All audit columns present

Processing: Products.csv
  Read 42 rows, 14 cols
  Written to: abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/b

## 6. Summary

In [18]:
summary = pd.DataFrame(results)
print(summary[["file", "status", "rows", "issues"]].to_string(index=False))

failed = summary[summary["status"] == "FAILED"]
if len(failed) > 0:
    raise Exception(f"{len(failed)} file(s) failed to load. Do not proceed to Silver.")
else:
    print("\nAll files ingested. Check issues column before running Silver notebook.")

              file  status  rows                                             issues
PurchaseOrders.csv SUCCESS  1248 FAIL: Required column 'PONumber' not found in file
   SalesOrders.csv SUCCESS  5150                        87 duplicate OrderID values
      Products.csv SUCCESS    42                       1 duplicate ProductID values
     Customers.csv SUCCESS    72                      1 duplicate CustomerID values
     Inventory.csv SUCCESS   215                     168 duplicate ProductID values

All files ingested. Check issues column before running Silver notebook.


## 7. Quick peek at output

In [19]:
# Verify each csv file was written and spot-check audit columns
for cfg in FILE_CONFIGS:
    path = OUTPUT_PATH + cfg["target"] + ".csv"
    if path:
        df_check = pd.read_csv(path)
        print(f"\n{cfg['target']}")
        print(f"  Rows    : {len(df_check):,}")
        print(f"  Columns : {len(df_check.columns)}")
        print(f"  Sample audit cols:")
        print(df_check[["_ingested_at", "_source_file", "_pipeline_run_id"]].head(2).to_string())
    else:
        print(f"{cfg['target']}: FILE NOT FOUND")


bronze_purchase_orders
  Rows    : 1,248
  Columns : 22
  Sample audit cols:
                 _ingested_at        _source_file _pipeline_run_id
0  2026-06-04 21:15:29.588755  PurchaseOrders.csv  20260604_211529
1  2026-06-04 21:15:29.588755  PurchaseOrders.csv  20260604_211529

bronze_sales_orders
  Rows    : 5,150
  Columns : 20
  Sample audit cols:
                 _ingested_at     _source_file _pipeline_run_id
0  2026-06-04 21:15:29.588755  SalesOrders.csv  20260604_211529
1  2026-06-04 21:15:29.588755  SalesOrders.csv  20260604_211529

bronze_products
  Rows    : 42
  Columns : 17
  Sample audit cols:
                 _ingested_at  _source_file _pipeline_run_id
0  2026-06-04 21:15:29.588755  Products.csv  20260604_211529
1  2026-06-04 21:15:29.588755  Products.csv  20260604_211529

bronze_customers
  Rows    : 72
  Columns : 20
  Sample audit cols:
                 _ingested_at   _source_file _pipeline_run_id
0  2026-06-04 21:15:29.588755  Customers.csv  20260604_211529
1  2026-06